# Ionospheric Forecasting Dataset: Example Notebook

This notebook demonstrates how to load and use the **ML-ready ionospheric forecasting dataset** released as part of NASA's Heliolab Frontier Development Lab (FDL) 2025 research program.

## What is the ionosphere and why does it matter?

The **ionosphere** is a region of Earth's upper atmosphere (roughly 60–1,000 km altitude) where solar radiation ionizes atoms and molecules, creating a layer of free electrons. This layer directly affects radio wave propagation and is critical for **GPS/GNSS accuracy** — signals passing through the ionosphere are delayed in proportion to the electron density along the signal path.

**Total Electron Content (TEC)** is the key quantity measured in this dataset. TEC represents the total number of electrons in a column between a satellite and a ground receiver, measured in TEC Units (1 TECU = 10¹⁶ electrons/m²). Higher TEC values mean greater signal delay and potential positioning errors.

## What is in this dataset?

This dataset integrates multiple heterogeneous data sources — all temporally aligned to a **15-minute cadence** — to support ML-based ionospheric nowcasting and forecasting:

| Dataset | Description | Format |
|---------|-------------|--------|
| **JPLD** | JPL Global Ionospheric Maps (GIMs) — global TEC maps at 1°×1° (180×360) resolution from GPS observations | WebDataset tar archives |
| **OMNIWeb** | Solar wind parameters and geomagnetic indices (AE, AL, SYM-H, Bx/By/Bz, velocity, etc.) | CSV |
| **CelesTrak** | Kp and Ap geomagnetic indices | CSV |
| **SunMoonGeometry** | Solar and lunar zenith angles, positions, and distances | Computed on-the-fly |
| **SET** | Solar Energetic Particle data | CSV |

For full details, see the paper: [Connecting the Dots: A Machine Learning Ready Dataset for Ionospheric Forecasting Models](https://doi.org/10.48550/arXiv.2511.15743).

# Example loading a batch of data using the PyTorch dataloader

This notebook is intended to run in [Google Colab](https://colab.research.google.com/github/FrontierDevelopmentLab/2025-HL-Ionosphere-dataset/blob/main/dataset_example_colab.ipynb), which has most of the dependencies pre-installed. 

<details>
<summary><b>Dataset directory structure</b> (click to expand)</summary>

The dataset is organized as follows (using wildcard/glob notation, as well as parentheses around year/month/day variables):

```
ionosphere-data-public
├── celestrak
│   ├── celestrak_SW-All.csv
│   ├── kp_ap_processed_timeseries.csv
│   └── kp_ap_processed_timeseries_deltamin_15_rewind_180.csv
├── google_android
│   ├── CAS0OPSRAP_20233090000_01D_01D_DCB.BIA.gz
│   ├── dcb_clusters.csv.gz
│   ├── LICENSE
│   ├── sat_xyz.csv.gz
│   ├── TLSE00FRA_R_20233090000_01D_30S_MO.crx.gz
│   ├── video_2023_09_11_to_2024_05_24.mp4
│   ├── vtec_maps
│   └── worldpop_population_per_km2_s2level10.csv.gz
├── jpld
│   ├── raw
│   │   └── (year)
│   │       └── jpld*.10i.nc.gz
│   └── webdataset
│       ├── dates_index_*
│       ├── jpld-*.tar
│       └── tar_files_index_3a8334f288afd9deb25c6ec57c6b0cbb
├── madrigal
│   ├── gps_tec_20120216.txt
│   ├── processed
│   │   ├── gps_data  
│   │   │   └── (2-digit year, e.g., 09)
│   │   │       └── gps_data_(4-digit-year).txt
│   │   └── gps_data_tarr
│   │       ├── (year)
│   │       │   └── (2-digit-month)
│   │       │       └── (2-digit-day)
│   │       │           └── *.tec_madrigal.npy
│   │       ├── csv_subsets
│   │       │   └── subset_tec_?0mln.csv
│   │       └── tars
│   │           └── dataset-*.tar
│   └── raw
│       ├── gps_data
│       │   └── gps*g.001.hdf5
│       ├── ne_dir
│       │   └── pfa*.001_lp_fit_*min.001.h5.gz
│       └── tec_dir
│           └── pfa*.001_lp_fit_*min.001.h5.gz
├── omniweb
│   ├── omniweb_indices_15min.csv
│   ├── omniweb_magnetic_field_15min.csv
│   └── omniweb_solar_wind_15min.csv
├── quasidipole
│   └── qd_lon_(year).npy
├── README.md
├── sdocore
│   └── sdo_core_dataset_21504.h5
└── set
    ├── karman-2025_data_sw_data_set_sw.csv
    └── karman-2025_data_sw_data_set_sw_deltamin_15_rewind_1440.csv
```

Data format is dependent on the data source: some are CSV tables (`*.csv`), some HDF5 hierarchical binary data (`*.h5`), some numpy arrays (`*.npy`), and some plain text files (`*.txt`), all in various forms of archive formats (`*.tar`, `*.tar.gz`, `*.gz`). 

</details>

To load the data, it is recommended to use the dedicated data loaders and processing scripts included in [this same GitHub repository](https://github.com/FrontierDevelopmentLab/2025-HL-Ionosphere-dataset.git) (see [Import section](##Import-Dedicated-Data-Loaders-and-Processing-Scripts) for examples). These data loaders and processing scripts load the data into memory and properly align it in time, so the data from the different sources can be used together.

For more details about the dataset, refer to the dedicated, descriptive, peer-reviewed journal article [Connecting the Dots: A Machine Learning Ready Dataset for Ionospheric Forecasting Models](https://doi.org/10.48550/arXiv.2511.15743).

In [ ]:
!pip install cartopy awscli skyfield
!pip install pandas<3

## Setup

Install required dependencies. [Cartopy](https://scitools.github.io/cartopy/) is used for geographic map projections, [Skyfield](https://rhodesmill.org/skyfield/) computes Sun/Moon positions, and `awscli` downloads data from the public AWS S3 bucket.

In [ ]:
# Download the data processing scripts (only needed on Colab)
import shutil
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists('scripts'):
    !git clone --filter=blob:none --sparse https://github.com/FrontierDevelopmentLab/2025-HL-Ionosphere-dataset.git
    !cd 2025-HL-Ionosphere-dataset && git sparse-checkout set scripts/
    shutil.move('2025-HL-Ionosphere-dataset/scripts', 'scripts')
    shutil.rmtree('2025-HL-Ionosphere-dataset')
elif not IN_COLAB:
    print('Not running on Colab — using local scripts/ directory')
else:
    print('scripts/ already exists, skipping clone')

The cell below clones only the `scripts/` directory from this repository. These scripts contain the dedicated **data loaders** that handle reading each data source format, applying normalization (e.g., log1p + z-score for TEC, Yeo-Johnson for solar wind), and aligning all sources to the common 15-minute time grid.

### Imports

In [ ]:
import os
import torch
import pickle
import hashlib
import tarfile
import datetime
import warnings
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import skyfield.api
from tqdm import tqdm
from glob import glob
from io import BytesIO
from functools import lru_cache
from torch.utils.data import Dataset

## Import Dedicated Data Loaders and Processing Scripts

In [ ]:
from scripts.events.events import EventCatalog

from scripts.datasets.dataset_jpld import JPLD
from scripts.datasets.dataset_sequences import Sequences
from scripts.datasets.dataset_union import Union
from scripts.datasets.dataset_sunmoongeometry import SunMoonGeometry
from scripts.datasets.dataset_quasidipole import QuasiDipole
from scripts.datasets.dataset_celestrak import CelesTrak
from scripts.datasets.dataset_omniweb import OMNIWeb, omniweb_all_columns
from scripts.datasets.dataset_set import SET, set_all_columns
from scripts.datasets.dataloader_cached import CachedDataLoader

## Download a sample of the dataset from AWS

The full dataset is hosted publicly on AWS S3 (no credentials required). Below we download:
- **OMNIWeb** CSV files (~few MB) — solar wind and geomagnetic index time series covering 1996–2024
- **JPLD** tar archives (~1 GB each) — each tar file contains several months of global TEC maps stored in WebDataset format, where each 15-minute snapshot is a 180×360 array

We download only 4 tar files here as a sample. The full JPLD dataset spans ~170+ tar files.

In [ ]:
import subprocess

s3_base = 's3://nasa-radiant-data/helioai-datasets/ionosphere-data-public'

# --- OMNIWeb ---
omniweb_files = [
    'omniweb_indices_15min.csv',
    'omniweb_magnetic_field_15min.csv',
    'omniweb_solar_wind_15min.csv',
]
os.makedirs('data/omniweb', exist_ok=True)
for f in omniweb_files:
    local_path = f'data/omniweb/{f}'
    if os.path.exists(local_path):
        print(f'Already exists, skipping: {local_path}')
    else:
        print(f'Downloading: {local_path}')
        subprocess.run(['aws', 's3', 'cp', '--no-sign-request', f'{s3_base}/omniweb/{f}', local_path], check=True)

# --- JPLD ---
os.makedirs('data/jpld', exist_ok=True)

# Tar files index
index_name = 'tar_files_index_3a8334f288afd9deb25c6ec57c6b0cbb'
index_path = f'data/jpld/{index_name}'
if os.path.exists(index_path):
    print(f'Already exists, skipping: {index_path}')
else:
    print(f'Downloading: {index_path}')
    subprocess.run(['aws', 's3', 'cp', '--no-sign-request', f'{s3_base}/jpld/webdataset/{index_name}', index_path], check=True)

# Download a sample of JPLD tar files (each ~1 GB)
for i in range(168, 172):
    tar_name = f'jpld-{i}.tar'
    local_path = f'data/jpld/{tar_name}'
    if os.path.exists(local_path):
        print(f'Already exists, skipping: {local_path}')
    else:
        print(f'Downloading: {local_path}')
        subprocess.run(['aws', 's3', 'cp', '--no-sign-request', f'{s3_base}/jpld/webdataset/{tar_name}', local_path], check=True)

## Dataset overview

The plot below shows a time series overview of the downloaded data: the global mean and max TEC from the JPLD maps alongside key space weather drivers from OMNIWeb. This gives a sense of what conditions are captured in the dataset — notably the **May 2024 G5 geomagnetic storm** (the strongest of solar cycle 25), visible as the large SYM-H dip, AE spike, southward IMF Bz excursion, and corresponding TEC enhancement.

In [ ]:
import matplotlib.dates as mdates

# Compute global mean and max TEC for each time step (sampled every 4 hours for speed)
dataset_jpld_all = JPLD('data/jpld/')
all_dates = dataset_jpld_all.dates
sample_step = 16  # every 16th 15-min step = every 4 hours
sampled_dates = all_dates[::sample_step]

tec_means, tec_maxs, tec_dates = [], [], []
for d in tqdm(sampled_dates, desc='Computing TEC stats'):
    tecmap, date_str = dataset_jpld_all[d]
    raw = JPLD.unnormalize(tecmap)
    tec_means.append(float(raw.mean()))
    tec_maxs.append(float(raw.max()))
    tec_dates.append(d)

# Load OMNIWeb data and filter to JPLD date range
df_idx = pd.read_csv('data/omniweb/omniweb_indices_15min.csv', parse_dates=['all__dates_datetime__'])
df_mag = pd.read_csv('data/omniweb/omniweb_magnetic_field_15min.csv', parse_dates=['all__dates_datetime__'])
df_sw  = pd.read_csv('data/omniweb/omniweb_solar_wind_15min.csv', parse_dates=['all__dates_datetime__'])

t0, t1 = min(tec_dates), max(tec_dates)
df_idx = df_idx[(df_idx['all__dates_datetime__'] >= t0) & (df_idx['all__dates_datetime__'] <= t1)]
df_mag = df_mag[(df_mag['all__dates_datetime__'] >= t0) & (df_mag['all__dates_datetime__'] <= t1)]
df_sw  = df_sw[(df_sw['all__dates_datetime__'] >= t0) & (df_sw['all__dates_datetime__'] <= t1)]

# Downsample OMNIWeb to 4-hour means for cleaner plotting
df_idx = df_idx.set_index('all__dates_datetime__').resample('4h').mean()
df_mag = df_mag.set_index('all__dates_datetime__').resample('4h').mean()
df_sw  = df_sw.set_index('all__dates_datetime__').resample('4h').mean()

# Plot
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

# Panel 1: Mean and Max TEC
ax = axes[0]
ax.plot(tec_dates, tec_means, color='C0', lw=0.8, label='Mean TEC')
ax.fill_between(tec_dates, tec_means, alpha=0.2, color='C0')
ax2 = ax.twinx()
ax2.plot(tec_dates, tec_maxs, color='C1', lw=0.8, alpha=0.7, label='Max TEC')
ax.set_ylabel('Mean TEC [TECU]')
ax2.set_ylabel('Max TEC [TECU]')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
ax.set_title('Dataset Overview: JPLD TEC Maps & Space Weather Drivers')

# Panel 2: SYM-H (storm indicator)
ax = axes[1]
ax.plot(df_idx.index, df_idx['omniweb__sym_h__[nT]'], color='C3', lw=0.8)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axhline(-50, color='red', lw=0.5, ls=':', alpha=0.5)
ax.set_ylabel('SYM-H [nT]')
ax.text(0.01, 0.05, 'storm threshold', transform=ax.transAxes, fontsize=7, color='red', alpha=0.7)

# Panel 3: AE index (substorm activity)
ax = axes[2]
ax.plot(df_idx.index, df_idx['omniweb__ae_index__[nT]'], color='C4', lw=0.8)
ax.set_ylabel('AE Index [nT]')

# Panel 4: IMF Bz (southward = geoeffective)
ax = axes[3]
ax.plot(df_mag.index, df_mag['omniweb__bz_gse__[nT]'], color='C2', lw=0.8)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.fill_between(df_mag.index, df_mag['omniweb__bz_gse__[nT]'], 0,
                where=df_mag['omniweb__bz_gse__[nT]'] < 0, alpha=0.15, color='red', label='Southward (geoeffective)')
ax.set_ylabel('IMF Bz [nT]')
ax.legend(fontsize=8, loc='lower right')

# Panel 5: Solar wind speed
ax = axes[4]
ax.plot(df_sw.index, df_sw['omniweb__speed__[km/s]'], color='C5', lw=0.8)
ax.set_ylabel('SW Speed [km/s]')
ax.set_xlabel('Date')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=0))

plt.tight_layout()
plt.show()

## Load a single batch of data

The simplest way to use the dataset is to load a single TEC map by date. The `JPLD` loader reads from the WebDataset tar archives and returns a normalized TEC map as a NumPy array of shape `(1, 180, 360)` — a single-channel image where each pixel represents the Total Electron Content at that latitude/longitude.

In [ ]:
# Create a dataset
dataset_jpld_dir = 'data/jpld'

date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)

dataset_jpld = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end)

In [ ]:
# Load the JPLD map corresponding to date datetime.datetime(2024, 5, 9, 18, 30)
date = datetime.datetime(2024, 5, 9, 18, 30)
tec_map, tec_date = dataset_jpld[date]
print(tec_map.shape, tec_date)
print(type(tec_date))

### Plot the TEC map

Visualize the global TEC map using a [Plate Carrée projection](https://en.wikipedia.org/wiki/Equirectangular_projection) with coastlines overlaid. Brighter regions indicate higher electron density — typically concentrated near the equator on the dayside of Earth due to solar irradiation, forming the equatorial ionization anomaly (EIA).

In [ ]:
# Plot the TEC map
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

im = ax.imshow(
    tec_map[0],
    extent=[-180, 180, -90, 90],
    origin='upper',
    cmap='jet',
    transform=ccrs.PlateCarree()
)

ax.coastlines()
ax.set_xlabel('Longitude [°]')
ax.set_ylabel('Latitude [°]')
ax.set_title(f'JPL Global Ionospheric Map — {tec_date} UT')

cbar = plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8, pad=0.05)
cbar.set_label('TEC [normalized]')

plt.tight_layout()
plt.show()

## Sequence of multiple datasets

For ML forecasting, we need **temporal sequences** of aligned multi-source data — not just single snapshots. The `Sequences` class combines multiple dataset loaders and produces sequences of a specified length with configurable **dilation** (temporal stride between steps).

In this example:
- **Context window:** 2 time steps of past data (input to the model)
- **Prediction window:** 1 time step to forecast (the target)
- **Dilation:** 16 × 15 min = 4 hours between each step in the sequence
- **Datasets combined:** JPLD (TEC maps), OMNIWeb (solar wind/geomagnetic indices), and SunMoonGeometry (solar zenith angles)

This means the model sees TEC maps and driving conditions at t=0h and t=4h, and must predict the TEC map at t=8h.

In [ ]:
# Initialize variables
date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)
image_size = (180, 360)
context_window = 2 # Number of time steps of context used in model
prediction_window = 1 # Number of time steps to predict
training_sequence_length = context_window + prediction_window
delta_minutes = 15 # 15-minute cadence
date_dilation = 16 # Use 16x dilation for 15-min data to get 4-hour context

date_exclusions = []
datasets_omniweb_valid = []
datasets_jpld_valid = []
datasets_sunmoon_valid = []

dataset_omniweb_dir = 'data/omniweb/' #'/path/to/omniweb/data/'
dataset_jpld_dir = 'data/jpld' #'/path/to/jpld/data/'

# Create sequence dataset excluding validation events
dataset_jpld = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)
dataset_omniweb = OMNIWeb(dataset_omniweb_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions, return_as_image_size=image_size)
dataset_sunmoon = SunMoonGeometry(date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)

# Create Sequence datasets
dataset = Sequences(datasets=[dataset_jpld, dataset_omniweb, dataset_sunmoon], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)


## Train and Validation setup

A key challenge in ionospheric forecasting is evaluating model performance during **geomagnetic storms** — the events that matter most for GPS accuracy and space weather impacts. Storms are categorized using the Mestici Scale (e.g., G5H3 = NOAA G5 storm at H3 intensity level).

The recommended approach is to **exclude storm events from training** and reserve them for validation/testing. This ensures the model is evaluated on the most operationally relevant conditions it hasn't seen during training.

Below, we demonstrate this event-based splitting strategy:
1. Define a storm event (the May 10, 2024 G5 storm — the strongest in solar cycle 25)
2. Exclude it (plus a context-window buffer before it) from the training set
3. Create a separate validation dataset covering only the storm period

In [ ]:
# Initialize variables
date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)
image_size = (180, 360)
context_window = 2 # Number of time steps of context used in model
prediction_window = 1 # Number of time steps to predict
training_sequence_length = context_window + prediction_window
delta_minutes = 15 # 15-minute cadence
date_dilation = 16 # Use 16x dilation for 15-min data to get 4-hour context

dataset_omniweb_dir = 'data/omniweb/' #'/path/to/omniweb/data/'
dataset_jpld_dir = 'data/jpld' #'/path/to/jpld/data/'

# Make a EventCatalog[event_id], a dict with keys:
# 'date_start': date_start,
# 'date_end': date_end,
# 'duration': duration,
# 'max_kp': max_kp,
# 'time_steps': time_steps
event_id = "G5H3-202405101500"
event_start = datetime.datetime(2024, 5, 10, 15, 0).isoformat()
event_end = datetime.datetime(2024, 5, 10, 18, 0).isoformat()
duration = datetime.datetime.fromisoformat(event_end) - datetime.datetime.fromisoformat(event_start)
max_kp = 9.0
time_steps = int(duration.total_seconds() / (15*60))
catalog = {}
catalog[event_id] = {
  'date_start': event_start,
  'date_end': event_end,
  'duration': duration,
  'max_kp': max_kp,
  'time_steps': time_steps
}

event_catalog = EventCatalog(catalog=catalog)
valid_event_id = ["G5H3-202405101500"]

date_exclusions = []
datasets_omniweb_valid = []
datasets_jpld_valid = []
datasets_sunmoon_valid = []

# Process validation events
for event_id in valid_event_id:
    print('Excluding event ID: {}'.format(event_id))

    if event_id not in event_catalog:
        raise ValueError('Event ID {} not found in EventCatalog'.format(event_id))

    event = event_catalog[event_id]
    exclusion_start = datetime.datetime.fromisoformat(event['date_start']) - datetime.timedelta(minutes=context_window * delta_minutes)
    exclusion_end = datetime.datetime.fromisoformat(event['date_end'])
    date_exclusions.append((exclusion_start, exclusion_end))
    print('Exclusion start: {}, end: {}'.format(exclusion_start, exclusion_end))

    datasets_omniweb_valid.append(OMNIWeb(dataset_omniweb_dir, date_start=exclusion_start, date_end=exclusion_end, return_as_image_size=image_size))
    datasets_jpld_valid.append(JPLD(dataset_jpld_dir, date_start=exclusion_start, date_end=exclusion_end))
    datasets_sunmoon_valid.append(SunMoonGeometry(date_start=exclusion_start, date_end=exclusion_end))

# Create validation dataset
dataset_jpld_valid = Union(datasets=datasets_jpld_valid)
dataset_omniweb_valid = Union(datasets=datasets_omniweb_valid)
dataset_sunmoon_valid = Union(datasets=datasets_sunmoon_valid)

# Create training dataset excluding validation events
dataset_jpld_train = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)
dataset_omniweb_train = OMNIWeb(dataset_omniweb_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions, return_as_image_size=image_size)
dataset_sunmoon_train = SunMoonGeometry(date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)

# Create Sequence datasets
dataset_valid = Sequences(datasets=[dataset_jpld_valid, dataset_omniweb_valid, dataset_sunmoon_valid], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)
dataset_train = Sequences(datasets=[dataset_jpld_train, dataset_omniweb_train, dataset_sunmoon_train], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)


# Q: What is one question that you have answered using these data? 

This dataset addresses the temporal relationship between ionospheric drivers (solar wind, interplanetary magnetic field strength, Kp, etc.) and Global Navigation Satellite Systems  derived global ionospheric maps (GIMs). It does this by integrating heterogeneous, multi-source observations and harmonizing temporal and spatial scales. This dataset provides the foundation upon which advanced machine learning architectures can be developed, tested, and benchmarked for global-scale ionospheric nowcasting and forecasting.

This work has been performed and completed (see the [code](https://github.com/FrontierDevelopmentLab/2025-HL-Ionosphere/) and peer-reviewed journal article [Forecasting the Ionosphere from Sparse GNSS Data with Temporal-Fusion Transformers](https://doi.org/10.48550/arXiv.2509.00631) for details).

# Q: What is one unanswered question that you think could be answered using these data? 

An unanswered question that remains from this dataset is the physical relationships between the various ionospheric drivers and GIMs. This dataset was used to train a machine learning model to forecast GIMs, but not to analyze the relationships between drivers and the resulting GIM. It would be interesting to in future address more than just the temporal relationships between these data.